# PSA Translation — NLLB-200-distilled-600M Fine-Tuning
### English/Kiswahili → Ekegusii (low-resource) machine translation

This notebook trains and evaluates **NLLB-200-distilled-600M** with layer freezing on a curated
English/Kiswahili → Ekegusii PSA (Public Service Announcement) dataset.

**No Colab, Kaggle, or Weights & Biases dependency** — designed to run on any standard
Python + GPU environment (e.g. Navon Cloud JupyterLab). All logs, metrics, and checkpoints
are written to local files under `./checkpoints/` and `./logs/`.

**Requirements:** Python 3.10+, a CUDA GPU (tested on NVIDIA T4 15GB; will run faster/larger
batches on an A100), and `Final_merged_psas.csv` placed in the same directory as this notebook
(or update `DATA_PATH` below).

**Note on memory settings below:** the batch size, gradient accumulation, gradient
checkpointing, and Adafactor optimizer choices were tuned on a 15GB T4 GPU to avoid
out-of-memory errors. On a larger GPU (e.g. an 80GB A100) these are conservative and can
likely be relaxed (e.g. larger batch size, Adam instead of Adafactor) for faster training —
but they are left as the tested, working configuration by default so this runs without
errors on any GPU size.


## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate


In [ ]:
import os
# Restrict to a single GPU by default -- safe no-op on single-GPU machines,
# and avoids a known multi-GPU memory-overhead issue on some multi-GPU setups.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")


In [ ]:
import os, json, time, random
import numpy as np
import pandas as pd
import torch

from transformers import set_seed

# Reproducibility
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Local output directories (created automatically, no cloud mounts needed)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)


## 2. Load curated dataset

In [ ]:
# Place Final_merged_psas.csv in the same folder as this notebook,
# or set DATA_PATH to its full path.
DATA_PATH = "Final_merged_psas.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()


### 2.1 Build the combined (English + Kiswahili) → Ekegusii dataset

Each row becomes **two** training examples where possible: one with English as source,
one with Kiswahili as source, both mapping to the same Ekegusii target.

In [ ]:
def build_combined(df):
    rows = []
    for _, r in df.iterrows():
        if pd.notna(r["English"]) and str(r["English"]).strip():
            rows.append({
                "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                "source_text": r["English"], "target_text": r["Ekegusii"],
                "source_lang": "en"
            })
        if pd.notna(r.get("Kiswahili")) and str(r.get("Kiswahili")).strip():
            rows.append({
                "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                "source_text": r["Kiswahili"], "target_text": r["Ekegusii"],
                "source_lang": "sw"
            })
    return pd.DataFrame(rows).dropna(subset=["target_text"])

combined = build_combined(df)
combined = combined[combined["target_text"].astype(str).str.strip() != ""]
print("Total combined examples:", len(combined))
print(combined["source_lang"].value_counts())
print(combined["Domain"].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

# stratify by source_lang so both directions are represented in every split
train_df, temp_df = train_test_split(combined, test_size=0.2, random_state=42,
                                      stratify=combined["source_lang"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42,
                                    stratify=temp_df["source_lang"])

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))
train_df["source_lang"].value_counts(), test_df["source_lang"].value_counts()


**Low-resource note:** Ekegusii has no native language code in NLLB-200 and is not
in its training data, so this is a genuine low-resource target. A placeholder target
language tag (`swh_Latn`, Kiswahili's code) is used to steer generation, since NLLB
requires a valid target-language token at generation time.

## 3. Shared utilities: tokenization, metrics, layer freezing, timing

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import evaluate

sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def to_hf(d):
    return Dataset.from_pandas(d[["source_text", "target_text", "source_lang", "Domain"]]
                                .reset_index(drop=True))

train_ds = to_hf(train_df)
val_ds   = to_hf(val_df)
test_ds  = to_hf(test_df)

def build_compute_metrics(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        # -100 is the label-ignore sentinel; must be swapped for a real pad id before decoding
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        bleu = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        c = chrf.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        return {"bleu": bleu["score"], "chrf": c["score"]}
    return compute_metrics

def freeze_encoder_layers(model, num_layers_to_freeze):
    """Freeze bottom N encoder layers to reduce overfitting risk on our small,
    low-resource fine-tuning set and cut compute cost."""
    encoder = model.get_encoder()
    layers = encoder.block if hasattr(encoder, "block") else encoder.layers
    for i, layer in enumerate(layers):
        if i < num_layers_to_freeze:
            for p in layer.parameters():
                p.requires_grad = False
    return model

def score(preds, refs, label):
    """Computes BLEU/chrF, prints, and returns a result dict (no external logging service)."""
    bleu = sacrebleu.compute(predictions=preds, references=[[r] for r in refs])
    c = chrf.compute(predictions=preds, references=[[r] for r in refs])
    print(f"{label:35s} BLEU={bleu['score']:.2f}  chrF={c['score']:.2f}")
    return {"name": label, "bleu": bleu["score"], "chrf": c["score"]}

MAX_LEN = 128
results_log = []          # collects every score() call for the final summary table
timing_log = {}           # collects wall-clock training time


## 4. NLLB-200-distilled-600M — tokenizer and preprocessing

In [ ]:
NLLB_CHECKPOINT = "facebook/nllb-200-distilled-600M"
TGT_PLACEHOLDER = "swh_Latn"  # placeholder tag for Ekegusii (unsupported by NLLB-200)

nllb_tok = AutoTokenizer.from_pretrained(NLLB_CHECKPOINT)

def nllb_src_code(source_lang):
    return "eng_Latn" if source_lang == "en" else "swh_Latn"

def preprocess_nllb(batch):
    all_ids, all_labels = [], []
    for sl, src, tgt in zip(batch["source_lang"], batch["source_text"], batch["target_text"]):
        nllb_tok.src_lang = nllb_src_code(sl)
        enc = nllb_tok(src, text_target=tgt, max_length=MAX_LEN, truncation=True)
        all_ids.append(enc)
        all_labels.append(enc["labels"])
    return {
        "input_ids": [e["input_ids"] for e in all_ids],
        "attention_mask": [e["attention_mask"] for e in all_ids],
        "labels": all_labels,
    }

train_tok_nllb = train_ds.map(preprocess_nllb, batched=True, batch_size=16)
val_tok_nllb   = val_ds.map(preprocess_nllb, batched=True, batch_size=16)


### 4.1 Baseline (zero-shot) — NLLB, before any fine-tuning

In [ ]:
base_nllb = AutoModelForSeq2SeqLM.from_pretrained(NLLB_CHECKPOINT)
base_nllb.to("cuda" if torch.cuda.is_available() else "cpu")

test_en = test_df[test_df["source_lang"] == "en"]
test_sw = test_df[test_df["source_lang"] == "sw"]

def generate_nllb(model, texts, source_langs, tgt_code=TGT_PLACEHOLDER):
    preds = []
    for sl, t in zip(source_langs, texts):
        nllb_tok.src_lang = nllb_src_code(sl)
        enc = nllb_tok(t, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        forced_bos = nllb_tok.convert_tokens_to_ids(tgt_code)
        out = model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN)
        preds.append(nllb_tok.decode(out[0], skip_special_tokens=True))
    return preds

# NLLB inference is per-example (language-tagged), so we subsample the test set for speed.
# Increase n_eval if you have more GPU time to spare.
n_eval = min(60, len(test_en))
preds_base_en_nllb = generate_nllb(base_nllb, list(test_en["source_text"])[:n_eval], list(test_en["source_lang"])[:n_eval])
results_log.append(score(preds_base_en_nllb, list(test_en["target_text"])[:n_eval], "nllb_zero-shot_en-guz"))

n_eval_sw = min(60, len(test_sw))
preds_base_sw_nllb = generate_nllb(base_nllb, list(test_sw["source_text"])[:n_eval_sw], list(test_sw["source_lang"])[:n_eval_sw])
results_log.append(score(preds_base_sw_nllb, list(test_sw["target_text"])[:n_eval_sw], "nllb_zero-shot_sw-guz"))

del base_nllb
if torch.cuda.is_available():
    torch.cuda.empty_cache()

with open("logs/nllb_results_so_far.json", "w") as f:
    json.dump(results_log, f, indent=2)


### 4.2 Fine-tuning — NLLB (few-shot), with layer freezing

Checkpoints save automatically each epoch to `checkpoints/nllb_combined_guz/`.
Training logs (loss, BLEU, chrF per epoch) are written to `logs/nllb_training_log.csv`
after training completes — no external logging service required.

Uses `Adafactor` (lower memory footprint than Adam) with gradient checkpointing and
gradient accumulation — see the note in the intro cell about relaxing these on a
larger GPU.

In [ ]:
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_CHECKPOINT)
nllb_model = freeze_encoder_layers(nllb_model, num_layers_to_freeze=6)  # heavier model -> freeze more

args_nllb = Seq2SeqTrainingArguments(
    output_dir="checkpoints/nllb_combined_guz",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-3,           # Adafactor generally needs a higher LR than Adam
    num_train_epochs=5,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=20,
    report_to="none",             # no external logging service
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    gradient_checkpointing=True,
    optim="adafactor",            # lower memory footprint than Adam
)

trainer_nllb = Seq2SeqTrainer(
    model=nllb_model,
    args=args_nllb,
    train_dataset=train_tok_nllb,
    eval_dataset=val_tok_nllb,
    data_collator=DataCollatorForSeq2Seq(nllb_tok, model=nllb_model),
    processing_class=nllb_tok,
    compute_metrics=build_compute_metrics(nllb_tok),
)

t0 = time.time()
trainer_nllb.train()
timing_log["nllb_train_seconds"] = time.time() - t0
print(f"NLLB training time: {timing_log['nllb_train_seconds']/60:.1f} minutes")

# Save the best checkpoint (load_best_model_at_end=True already loaded it into memory) as a
# clean, inference-ready model -- this is what the demo function below loads.
trainer_nllb.save_model("checkpoints/nllb_combined_guz/final")
nllb_tok.save_pretrained("checkpoints/nllb_combined_guz/final")

# --- Save training log locally (replaces the W&B dashboard) ---
log_df = pd.DataFrame(trainer_nllb.state.log_history)
log_df.to_csv("logs/nllb_training_log.csv", index=False)
print("Training log saved to logs/nllb_training_log.csv")


### 4.3 Fine-tuned evaluation — NLLB, per source language

In [ ]:
preds_ft_en_nllb = generate_nllb(trainer_nllb.model, list(test_en["source_text"])[:n_eval], list(test_en["source_lang"])[:n_eval])
results_log.append(score(preds_ft_en_nllb, list(test_en["target_text"])[:n_eval], "nllb_few-shot_en-guz"))

preds_ft_sw_nllb = generate_nllb(trainer_nllb.model, list(test_sw["source_text"])[:n_eval_sw], list(test_sw["source_lang"])[:n_eval_sw])
results_log.append(score(preds_ft_sw_nllb, list(test_sw["target_text"])[:n_eval_sw], "nllb_few-shot_sw-guz"))

# Persist all results (zero-shot + few-shot) locally
with open("logs/nllb_results.json", "w") as f:
    json.dump(results_log, f, indent=2)
print("Results saved to logs/nllb_results.json")


## 5. Hyperparameters, training time, and results

Auto-generated from what actually ran, and saved to `logs/nllb_hyperparameters.csv`
and `logs/nllb_results_table.csv` for the write-up.

In [ ]:
hyperparam_table = pd.DataFrame([{
    "Model": "NLLB-200-distilled-600M",
    "Pair": "combined (en+sw)->guz",
    "Epochs": args_nllb.num_train_epochs,
    "Batch size": f"{args_nllb.per_device_train_batch_size} (x{args_nllb.gradient_accumulation_steps} accum)",
    "Learning rate": args_nllb.learning_rate,
    "Frozen encoder layers": "6/12",
    "Optimizer": args_nllb.optim,
    "fp16": args_nllb.fp16,
    "Gradient checkpointing": args_nllb.gradient_checkpointing,
    "Train time (min)": round(timing_log.get("nllb_train_seconds", 0) / 60, 1),
}])
hyperparam_table.to_csv("logs/nllb_hyperparameters.csv", index=False)
hyperparam_table


In [ ]:
results_df = pd.DataFrame(results_log)
results_df.to_csv("logs/nllb_results_table.csv", index=False)
print("=== Zero-shot vs Few-shot results ===")
print(results_df.to_string(index=False))


## 6. Inference demo

Loads the fine-tuned model from `checkpoints/nllb_combined_guz/final` and translates
sample sentences. Run this cell independently (after training, or in a fresh session
that has skipped straight to this section) to demonstrate translation on new input.

In [ ]:
FINAL_MODEL_DIR = "checkpoints/nllb_combined_guz/final"

# If this cell is run standalone (e.g. a fresh kernel after training already happened),
# reload the fine-tuned model + tokenizer from disk instead of relying on in-memory objects.
if "trainer_nllb" not in globals():
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    nllb_tok = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
    _nllb_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_MODEL_DIR)
    _nllb_model.to("cuda" if torch.cuda.is_available() else "cpu")
else:
    _nllb_model = trainer_nllb.model

TGT_PLACEHOLDER = "swh_Latn"

def nllb_src_code(source_lang):
    return "eng_Latn" if source_lang == "en" else "swh_Latn"

def translate_psa(text, source_lang="en"):
    """
    text: input sentence (English or Kiswahili)
    source_lang: 'en' (English) or 'sw' (Kiswahili)
    Returns: Ekegusii translation from the fine-tuned NLLB model
    """
    nllb_tok.src_lang = nllb_src_code(source_lang)
    inputs = nllb_tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(_nllb_model.device)
    forced_bos = nllb_tok.convert_tokens_to_ids(TGT_PLACEHOLDER)
    out = _nllb_model.generate(**inputs, forced_bos_token_id=forced_bos, max_length=MAX_LEN)
    return nllb_tok.decode(out[0], skip_special_tokens=True)

# --- Demo: sample PSAs ---
samples = [
    ("Farmers are urged to prioritize safe agrochemical usage this season.", "en"),
    ("Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.", "sw"),
]

for text, lang in samples:
    print(f"[{lang}] {text}")
    print("  NLLB ->", translate_psa(text, source_lang=lang))
    print()
